# WR Final Models — Reference Architectures

Jedinstveni notebook koji sadrzi **cista, finalna arhitektura** za svaki model iz WR pipeline-a.

**Ovaj notebook nije eksperimentalni.** Ovde nema Optune, grid searcha, ablacija, niti varijanti.
Svi izbori hiperparametara su fiksirani iz ranijih eksperimenata. Za kontekst i put kojim
su hiperparametri dobijeni, videti:

- `WR_Feature_Selection.ipynb` — kako je izabran top-40
- `WR_MLP_Comparison.ipynb`, `WR_MLP_Hybrid.ipynb`, `WR_MLP_Quantile.ipynb` — MLP varijante
- `WR_RNN_Improved.ipynb`, `WR_RNN_Improved_GRU.ipynb`, `WR_RNN_Attention_PlayerEmbed_v2.ipynb` — sekvencni
- `WR_Career_RNN_Optuna.ipynb` — RF/XGB/LGB baselines
- `WR_Ensemble_Final.ipynb` — konsolidacija + ensemble
- `docs/WR_kompletna_analiza.md` — ceo razvojni put i rezultati

## Modeli u ovom notebook-u

**Tabular (top-40 features, sqrt target, sample weights=0.6):**
1. RandomForest
2. XGBoost
3. LightGBM
4. ElasticNet
5. MLP Hybrid (Huber + GaussianNoise)
6. MLP Quantile q50 (pinball loss)

**Sekvencni (padded sequences, player embeddings):**
7. BiGRU + AttentionPool + PlayerEmbed (log1p target)
8. BiLSTM + AttentionPool + PlayerEmbed (log1p target)

## Uniformni recept
- Temporal split: train 2015–21, val 2022–23, test 2024–25
- Target: `sqrt(receiving_yards)` (tabular) / `log1p(receiving_yards)` (sekvencni)
- Sample weights: `1 + 0.6 * sqrt(y / mean_y)` (tabular), strength 0.7 za sekvencne
- Seed: 42


---
## 1. Imports


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, json, time, random
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, losses, Model, Input, regularizers
from tensorflow.keras.layers import (
    Dense, Dropout, LayerNormalization, GaussianNoise,
    LSTM, GRU, Bidirectional, Masking, Concatenate, Embedding, Flatten,
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['figure.figsize'] = (14, 5)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'TensorFlow {tf.__version__}')


TensorFlow 2.21.0


---
## 2. Feature Engineering (canonical pipeline)

Identicna priprema kao u `WR_Ensemble_Final` / `WR_RNN_Attention_PlayerEmbed_v2`.


In [ ]:
df = pd.read_csv('../data/fully combined/wr_all_weeks.csv')
df['week'] = df['game_id'].str.split('_').str[1].astype(int)
df = df.sort_values(['receiver_player_id', 'season', 'week']).reset_index(drop=True)

work_df = df.copy()

wp_rename_map = {
    'yards_wp_<25':       'yards_wp_less_than_25',
    'yards_wp_>75':       'yards_wp_greater_than_75',
    'receptions_wp_<25':  'receptions_wp_less_than_25',
    'receptions_wp_>75':  'receptions_wp_greater_than_75',
    'targets_wp_<25':     'targets_wp_less_than_25',
    'targets_wp_>75':     'targets_wp_greater_than_75',
}
work_df = work_df.rename(columns={k: v for k, v in wp_rename_map.items() if k in work_df.columns})

work_df['player_team_inferred'] = np.where(
    (work_df['home_team'] == work_df['defteam']) & (work_df['away_team'] != work_df['defteam']),
    work_df['away_team'],
    np.where(
        (work_df['away_team'] == work_df['defteam']) & (work_df['home_team'] != work_df['defteam']),
        work_df['home_team'], np.nan,
    ),
)
prev_team = work_df.groupby('receiver_player_id')['player_team_inferred'].shift(1)
work_df['team_changed'] = (
    work_df['player_team_inferred'].notna()
    & prev_team.notna()
    & (work_df['player_team_inferred'] != prev_team)
).astype(int)

prev_season = work_df.groupby('receiver_player_id')['season'].shift(1)
work_df['is_new_season'] = (prev_season.notna() & (work_df['season'] != prev_season)).astype(int)

work_df['season_week_abs'] = (work_df['season'] - 2015) * 22 + work_df['week']
work_df['weeks_since_last_game'] = (
    work_df.groupby('receiver_player_id')['season_week_abs']
    .diff().fillna(1).clip(lower=1).astype(float)
)
work_df.drop(columns=['season_week_abs'], inplace=True)

season_career = (
    work_df.groupby(['receiver_player_id', 'season'], as_index=False)
    .agg(avg_yards=('receiving_yards', 'mean'),
         avg_target_share=('target_share', 'mean'),
         avg_epa=('epa', 'mean'),
         avg_air_yard_share=('air_yard_share', 'mean'),
         avg_catch_rate=('catch_rate', 'mean'),
         games_played=('game_id', 'count'))
    .sort_values(['receiver_player_id', 'season'])
)
for c in ['avg_yards', 'avg_target_share', 'avg_epa',
          'avg_air_yard_share', 'avg_catch_rate', 'games_played']:
    season_career[f'{c}_last_season'] = (
        season_career.groupby('receiver_player_id')[c].shift(1)
    )
career_cols = [f'{c}_last_season' for c in
               ['avg_yards', 'avg_target_share', 'avg_epa',
                'avg_air_yard_share', 'avg_catch_rate', 'games_played']]
season_career = season_career[['receiver_player_id', 'season'] + career_cols]
work_df = work_df.merge(season_career, on=['receiver_player_id', 'season'], how='left')
work_df[career_cols] = work_df[career_cols].fillna(0)

# ------------------------------------------------------------------
# Full Ensemble_Final rolling/lag pipeline (produces top-40 features)
# ------------------------------------------------------------------
rolling_source_cols = [
    'targets', 'receptions', 'air_yards', 'yac', 'tds', 'epa', 'wpa', 'catch_rate',
    'avg_depth', 'adot', 'yac_per_reception', 'td_rate', 'explosive_plays', 'first_downs',
    'yards_per_target', 'team_pass_attempts', 'team_air_yards', 'team_epa', 'air_yard_share',
    'target_share', 'qb_completions', 'qb_attempts', 'qb_air_yards', 'qb_cpoe', 'qb_comp_pct',
    'avg_score_diff', 'trailing_pct', 'leading_pct', 'avg_quarter', 'success_rate',
    'big_play_rate', 'avg_start_yardline', 'red_zone_targets', 'end_zone_targets',
    'third_down_targets', 'fourth_down_targets', 'high_leverage_targets',
    'second_and_long_targets', 'third_and_medium_targets', 'wp_var', 'target_share_std',
    'reception_std', 'def_targets_dev', 'def_receptions_dev', 'def_yards_dev', 'def_tds_dev',
    'def_epa_dev', 'yards_Q1', 'yards_Q2', 'yards_Q3', 'yards_Q4',
    'receptions_Q1', 'receptions_Q2', 'receptions_Q3', 'receptions_Q4',
    'targets_Q1', 'targets_Q2', 'targets_Q3', 'targets_Q4',
    'lost_yards_due_to_penalty',
    'yards_wp_less_than_25', 'yards_wp_25_45', 'yards_wp_45_55',
    'yards_wp_55_75', 'yards_wp_greater_than_75',
    'receptions_wp_less_than_25', 'receptions_wp_25_45', 'receptions_wp_45_55',
    'receptions_wp_55_75', 'receptions_wp_greater_than_75',
    'targets_wp_less_than_25', 'targets_wp_25_45', 'targets_wp_45_55',
    'targets_wp_55_75', 'targets_wp_greater_than_75',
    'weeks_since_last_game',
]
available_roll_cols = [c for c in rolling_source_cols if c in work_df.columns]
grp = work_df.groupby('receiver_player_id', sort=False)
derived_cols = []
for col in available_roll_cols:
    work_df[f'{col}_lag1']  = grp[col].transform(lambda s: s.shift(1))
    work_df[f'{col}_roll5'] = grp[col].transform(
        lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
    )
    derived_cols.extend([f'{col}_lag1', f'{col}_roll5'])
# NOTE: raw rolling cols are NOT dropped - sequence branch needs them.

# Extra _roll3 + _lag1 on target + 5 base cols (used by sequence branch)
temporal_base = ['receiving_yards', 'targets', 'receptions',
                 'epa', 'target_share', 'air_yard_share']
for col in temporal_base:
    if col not in work_df.columns:
        continue
    if f'{col}_lag1' not in work_df.columns:
        work_df[f'{col}_lag1'] = grp[col].transform(lambda s: s.shift(1))
    work_df[f'{col}_roll3'] = grp[col].transform(
        lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
    )

# Momentum features (lag1 - roll5)
momentum_sources = [
    'targets', 'receptions', 'air_yards', 'epa', 'catch_rate',
    'target_share', 'yards_per_target', 'air_yard_share',
]
momentum_cols = []
for col in momentum_sources:
    lag1_col, roll5_col = f'{col}_lag1', f'{col}_roll5'
    if lag1_col in work_df.columns and roll5_col in work_df.columns:
        mcol = f'{col}_momentum'
        work_df[mcol] = work_df[lag1_col] - work_df[roll5_col]
        momentum_cols.append(mcol)

# Interaction features
interaction_cols = []
if 'target_share_lag1' in work_df.columns:
    work_df['target_volume_interaction'] = (
        work_df['target_share_lag1'] * work_df['pregame_total']
    )
    interaction_cols.append('target_volume_interaction')
if 'air_yard_share_lag1' in work_df.columns and 'team_pass_attempts_lag1' in work_df.columns:
    work_df['air_yards_expected'] = (
        work_df['air_yard_share_lag1'] * work_df['team_pass_attempts_lag1']
    )
    interaction_cols.append('air_yards_expected')
if 'catch_rate_lag1' in work_df.columns and 'targets_lag1' in work_df.columns:
    work_df['expected_receptions'] = (
        work_df['catch_rate_lag1'] * work_df['targets_lag1']
    )
    interaction_cols.append('expected_receptions')
if 'epa_lag1' in work_df.columns and 'targets_lag1' in work_df.columns:
    work_df['epa_per_target'] = (
        work_df['epa_lag1'] / work_df['targets_lag1'].replace(0, np.nan)
    ).fillna(0)
    interaction_cols.append('epa_per_target')

pregame_features = [
    'pregame_spread', 'pregame_total', 'surface', 'is_dome', 'temp_f',
    'humidity_pct', 'wind_mph', 'is_rain', 'is_snow', 'is_clear',
    'season', 'week', 'team_changed', 'is_new_season',
]
all_feature_columns = (
    pregame_features + career_cols + derived_cols + momentum_cols + interaction_cols
)

# Keep raw cols in model_df too so the sequence section can read them
model_df = work_df.copy()
lag1_all = [c for c in all_feature_columns if c.endswith('_lag1')]
model_df = model_df.loc[~model_df[lag1_all].isna().all(axis=1)].copy()
model_df = model_df.loc[:, ~model_df.columns.duplicated()]
model_df[all_feature_columns] = model_df[all_feature_columns].fillna(0)
model_df['receiving_yards_sqrt'] = np.sqrt(model_df['receiving_yards'].clip(lower=0))

print(f'Dataset: {model_df.shape}')
print(f'Tabular features engineered: {len(all_feature_columns)}')


Engineered dataset: (46115, 123)


---
## 3. Tabular Split + Top-40 Features + Sample Weights


In [3]:
train_seasons = list(range(2015, 2022))
val_seasons = [2022, 2023]
test_seasons = [2024, 2025]

train_df = model_df[model_df['season'].isin(train_seasons)].copy()
val_df   = model_df[model_df['season'].isin(val_seasons)].copy()
test_df  = model_df[model_df['season'].isin(test_seasons)].copy()

y_train_sqrt = train_df['receiving_yards_sqrt'].values
y_val_sqrt   = val_df['receiving_yards_sqrt'].values
y_test_sqrt  = test_df['receiving_yards_sqrt'].values
y_train_orig = train_df['receiving_yards'].values
y_val_orig   = val_df['receiving_yards'].values
y_test_orig  = test_df['receiving_yards'].values

with open('../results/selected_features_top40.json', 'r') as f:
    selected_features = json.load(f)['selected_features']

missing = [f for f in selected_features if f not in train_df.columns]
if missing:
    raise ValueError(f'Missing top-40 features: {missing}')

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[selected_features].values)
X_val   = scaler.transform(val_df[selected_features].values)
X_test  = scaler.transform(test_df[selected_features].values)

mean_y_train = np.mean(np.clip(y_train_orig, 0, None))
sw_train = 1.0 + 0.6 * np.sqrt(np.clip(y_train_orig, 0, None) / mean_y_train)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Features: {X_train.shape[1]} (top-40)')
print(f'Sample weights range: [{sw_train.min():.2f}, {sw_train.max():.2f}]')


ValueError: Missing top-40 features: ['first_downs_roll5', 'target_share_std_lag1', 'targets_roll5', 'air_yards_roll5', 'avg_start_yardline_roll5', 'target_share_std_roll5', 'air_yard_share_roll5', 'target_share_momentum', 'wp_var_lag1', 'qb_attempts_roll5', 'avg_depth_roll5', 'yards_Q1_roll5', 'team_air_yards_lag1', 'yards_wp_55_75_roll5', 'avg_score_diff_lag1', 'target_volume_interaction', 'air_yards_momentum', 'yards_Q2_roll5', 'reception_std_roll5', 'qb_completions_roll5', 'avg_start_yardline_lag1', 'qb_comp_pct_roll5', 'yards_wp_less_than_25_roll5', 'targets_wp_55_75_roll5', 'wpa_roll5', 'yards_Q3_roll5', 'receptions_momentum', 'yards_wp_55_75_lag1']

---
## 4. Utility Functions


In [ ]:
def eval_preds(y_true_orig, y_pred_orig):
    return {
        'MAE':  float(mean_absolute_error(y_true_orig, y_pred_orig)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))),
        'R2':   float(r2_score(y_true_orig, y_pred_orig)),
    }

def sqrt_to_orig(p):
    return np.clip(p, 0, None) ** 2

def log1p_to_orig(p):
    return np.expm1(np.clip(p, 0, None))

results = {}

def log_result(name, val_metrics, test_metrics, seconds=None):
    results[name] = {'val': val_metrics, 'test': test_metrics, 'seconds': seconds}
    s = f' ({seconds:.0f}s)' if seconds is not None else ''
    print(f'{name:32}{s}  val MAE={val_metrics["MAE"]:.3f}  '
          f'test MAE={test_metrics["MAE"]:.3f}  R2={test_metrics["R2"]:.4f}')

print('Utils defined.')


---
## 5. Tabular Models


### 5.1 RandomForest

Iz `WR_Career_RNN_Optuna` (najbolji tree baseline po MAE).


In [ ]:
t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=4,
    random_state=SEED, n_jobs=-1,
)
rf.fit(X_train, y_train_sqrt, sample_weight=sw_train)

rf_val_pred  = sqrt_to_orig(rf.predict(X_val))
rf_test_pred = sqrt_to_orig(rf.predict(X_test))
log_result('RandomForest',
           eval_preds(y_val_orig,  rf_val_pred),
           eval_preds(y_test_orig, rf_test_pred),
           time.time() - t0)


### 5.2 XGBoost


In [ ]:
t0 = time.time()
xgb = XGBRegressor(
    n_estimators=800, max_depth=6, learning_rate=0.03,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=4, gamma=0.1,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, tree_method='hist', verbosity=0,
)
xgb.fit(X_train, y_train_sqrt, sample_weight=sw_train,
        eval_set=[(X_val, y_val_sqrt)], verbose=False)

xgb_val_pred  = sqrt_to_orig(xgb.predict(X_val))
xgb_test_pred = sqrt_to_orig(xgb.predict(X_test))
log_result('XGBoost',
           eval_preds(y_val_orig,  xgb_val_pred),
           eval_preds(y_test_orig, xgb_test_pred),
           time.time() - t0)


### 5.3 LightGBM


In [ ]:
t0 = time.time()
lgb = LGBMRegressor(
    n_estimators=1000, num_leaves=63, learning_rate=0.03,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1,
    random_state=SEED, verbosity=-1,
)
lgb.fit(X_train, y_train_sqrt, sample_weight=sw_train,
        eval_set=[(X_val, y_val_sqrt)])

lgb_val_pred  = sqrt_to_orig(lgb.predict(X_val))
lgb_test_pred = sqrt_to_orig(lgb.predict(X_test))
log_result('LightGBM',
           eval_preds(y_val_orig,  lgb_val_pred),
           eval_preds(y_test_orig, lgb_test_pred),
           time.time() - t0)


### 5.4 ElasticNet


In [ ]:
t0 = time.time()
enet = ElasticNet(alpha=0.01, l1_ratio=0.3, max_iter=20000, random_state=SEED)
enet.fit(X_train, y_train_sqrt, sample_weight=sw_train)

enet_val_pred  = sqrt_to_orig(enet.predict(X_val))
enet_test_pred = sqrt_to_orig(enet.predict(X_test))
log_result('ElasticNet',
           eval_preds(y_val_orig,  enet_val_pred),
           eval_preds(y_test_orig, enet_test_pred),
           time.time() - t0)


### 5.5 MLP Hybrid (Huber)

MODEL_B_PARAMS iz `WR_MLP_Hybrid` + `GaussianNoise(0.15)`.
Arhitektura: `[448, 128, 320, 448, 256]`, Dropout 0.5, LayerNorm, AdamW, Huber loss.


In [ ]:
MODEL_B_PARAMS = {
    'n_layers': 5,
    'units': [448, 128, 320, 448, 256],
    'dropout': 0.5,
    'huber_delta': 0.5,
    'lr': 0.003701177981657943,
    'weight_decay': 1.5007044511603625e-05,
    'batch_size': 32,
}

def build_hybrid_mlp(input_dim, params, loss_fn, noise_stddev=0.15):
    model = keras.Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(GaussianNoise(noise_stddev))
    for i in range(params['n_layers']):
        model.add(Dense(params['units'][i], activation='relu'))
        model.add(LayerNormalization())
        dr = params['dropout'] * (0.5 if i >= params['n_layers'] - 1 else 1.0)
        model.add(Dropout(dr))
    model.add(Dense(1))
    opt = optimizers.AdamW(learning_rate=params['lr'],
                           weight_decay=params['weight_decay'])
    model.compile(optimizer=opt, loss=loss_fn, metrics=['mae'])
    return model


t0 = time.time()
tf.keras.backend.clear_session()
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

mlp_h = build_hybrid_mlp(X_train.shape[1], MODEL_B_PARAMS,
                         losses.Huber(delta=MODEL_B_PARAMS['huber_delta']))

cb_h = [
    callbacks.EarlyStopping(monitor='val_loss', patience=30,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                patience=10, min_lr=1e-6),
]

hist_h = mlp_h.fit(
    X_train, y_train_sqrt, sample_weight=sw_train,
    validation_data=(X_val, y_val_sqrt),
    epochs=500, batch_size=MODEL_B_PARAMS['batch_size'],
    callbacks=cb_h, verbose=0,
)

mlp_h_val_pred  = sqrt_to_orig(mlp_h.predict(X_val,  verbose=0).flatten())
mlp_h_test_pred = sqrt_to_orig(mlp_h.predict(X_test, verbose=0).flatten())
log_result('MLP Hybrid (Huber)',
           eval_preds(y_val_orig,  mlp_h_val_pred),
           eval_preds(y_test_orig, mlp_h_test_pred),
           time.time() - t0)


### 5.6 MLP Quantile q50 (pinball loss)

Ista `build_hybrid_mlp` arhitektura, ali sa pinball loss-om za tau=0.5 (medijan).
Najbolji single-model MAE u projektu (17.75).


In [ ]:
def pinball_loss_q50(y_true, y_pred):
    e = y_true - y_pred
    return tf.reduce_mean(tf.maximum(0.5 * e, (0.5 - 1.0) * e))


t0 = time.time()
tf.keras.backend.clear_session()
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

mlp_q = build_hybrid_mlp(X_train.shape[1], MODEL_B_PARAMS, pinball_loss_q50)

cb_q = [
    callbacks.EarlyStopping(monitor='val_loss', patience=30,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                patience=10, min_lr=1e-6),
]

hist_q = mlp_q.fit(
    X_train, y_train_sqrt, sample_weight=sw_train,
    validation_data=(X_val, y_val_sqrt),
    epochs=500, batch_size=MODEL_B_PARAMS['batch_size'],
    callbacks=cb_q, verbose=0,
)

mlp_q_val_pred  = sqrt_to_orig(mlp_q.predict(X_val,  verbose=0).flatten())
mlp_q_test_pred = sqrt_to_orig(mlp_q.predict(X_test, verbose=0).flatten())
log_result('MLP Quantile q50',
           eval_preds(y_val_orig,  mlp_q_val_pred),
           eval_preds(y_test_orig, mlp_q_test_pred),
           time.time() - t0)


---
## 6. Sequence Data Preparation

Dodatna priprema za sekvencne modele: padded career sequences + player embedding indeksi
+ static feature branch. Identicno `WR_RNN_Attention_PlayerEmbed_v2`.


In [ ]:
seq_raw_cols = [
    'receiving_yards', 'targets', 'receptions', 'air_yards', 'yac',
    'first_downs', 'tds', 'epa', 'wpa',
    'catch_rate', 'yards_per_target', 'avg_depth', 'adot', 'success_rate',
    'target_share', 'air_yard_share', 'target_share_std', 'reception_std',
    'team_pass_attempts', 'team_air_yards', 'team_epa',
    'qb_completions', 'qb_attempts', 'qb_comp_pct', 'qb_cpoe',
    'avg_score_diff', 'wp_var',
    'yards_Q1', 'yards_Q2', 'yards_Q3',
    'def_yards_dev', 'def_epa_dev',
    'weeks_since_last_game',
]
lag_cols  = [f'{c}_lag1'  for c in temporal_base]
roll_cols = [f'{c}_roll3' for c in temporal_base]
seq_feature_cols = seq_raw_cols + lag_cols + roll_cols

static_feature_cols = [
    'pregame_spread', 'pregame_total',
    'surface', 'is_dome', 'temp_f', 'humidity_pct', 'wind_mph',
    'is_rain', 'is_snow', 'is_clear',
    'week', 'team_changed', 'is_new_season', 'weeks_since_last_game',
    'avg_yards_last_season', 'avg_target_share_last_season',
    'avg_epa_last_season', 'avg_air_yard_share_last_season',
    'avg_catch_rate_last_season', 'games_played_last_season',
]

n_seq_features = len(seq_feature_cols)
n_static_features = len(static_feature_cols)
print(f'Sequence channels: {n_seq_features}   Static: {n_static_features}')


In [ ]:
# Player index (OOV = 0, train players get 1..N)
train_players = model_df.loc[model_df['season'].isin(train_seasons),
                             'receiver_player_id'].unique()
player_to_idx = {pid: i + 1 for i, pid in enumerate(sorted(train_players))}
n_players = len(player_to_idx) + 1
model_df['player_idx'] = (
    model_df['receiver_player_id'].map(player_to_idx).fillna(0).astype(int)
)

# Fill NaNs + fit scalers on train only
model_df[seq_feature_cols]    = model_df[seq_feature_cols].fillna(0)
model_df[static_feature_cols] = model_df[static_feature_cols].fillna(0)

train_mask = model_df['season'].isin(train_seasons)
seq_scaler    = StandardScaler().fit(model_df.loc[train_mask, seq_feature_cols])
static_scaler = StandardScaler().fit(model_df.loc[train_mask, static_feature_cols])

df_scaled = model_df.copy()
df_scaled[seq_feature_cols]    = seq_scaler.transform(model_df[seq_feature_cols])
df_scaled[static_feature_cols] = static_scaler.transform(model_df[static_feature_cols])
df_scaled['receiving_yards_orig'] = model_df['receiving_yards'].values

print(f'n_players = {n_players} (incl. OOV=0)')


In [ ]:
SEQ_LEN = 6

def build_sequences(df_scaled, seq_feats, static_feats, seq_len):
    X_seq, X_static, X_pid, y, seasons_arr = [], [], [], [], []
    for pid, group in df_scaled.groupby('receiver_player_id'):
        group = group.sort_values(['season', 'week'])
        if len(group) < 2:
            continue
        seq_vals    = group[seq_feats].values
        static_vals = group[static_feats].values
        yards       = group['receiving_yards_orig'].values
        season_vals = group['season'].values
        pid_vals    = group['player_idx'].values
        for t in range(1, len(group)):
            start = max(0, t - seq_len)
            past = seq_vals[start:t]
            if past.shape[0] < seq_len:
                pad = np.zeros((seq_len - past.shape[0], len(seq_feats)))
                past = np.vstack([pad, past])
            X_seq.append(past)
            X_static.append(static_vals[t])
            X_pid.append(pid_vals[t])
            y.append(yards[t])
            seasons_arr.append(season_vals[t])
    return (
        np.array(X_seq,    dtype=np.float32),
        np.array(X_static, dtype=np.float32),
        np.array(X_pid,    dtype=np.int32),
        np.array(y,        dtype=np.float32),
        np.array(seasons_arr),
    )

t0 = time.time()
X_seq_all, X_static_all, X_pid_all, y_all, seasons_all = build_sequences(
    df_scaled, seq_feature_cols, static_feature_cols, SEQ_LEN,
)
print(f'Built {len(X_seq_all)} sequences in {time.time()-t0:.1f}s')
print(f'  seq={X_seq_all.shape}  static={X_static_all.shape}')


In [ ]:
tr = np.isin(seasons_all, train_seasons)
va = np.isin(seasons_all, val_seasons)
te = np.isin(seasons_all, test_seasons)

X_seq_tr,    X_seq_va,    X_seq_te    = X_seq_all[tr],    X_seq_all[va],    X_seq_all[te]
X_static_tr, X_static_va, X_static_te = X_static_all[tr], X_static_all[va], X_static_all[te]
X_pid_tr,    X_pid_va,    X_pid_te    = X_pid_all[tr],    X_pid_all[va],    X_pid_all[te]
y_tr,        y_va,        y_te        = y_all[tr],        y_all[va],        y_all[te]

# log1p target (best transform for the attention RNN)
y_tr_log = np.log1p(np.clip(y_tr, 0, None))
y_va_log = np.log1p(np.clip(y_va, 0, None))

# sample weights for sequence models (strength 0.7 per v2)
mean_y_seq = np.mean(np.clip(y_tr, 0, None))
sw_tr_seq  = 1.0 + 0.7 * np.sqrt(np.clip(y_tr, 0, None) / mean_y_seq)

print(f'seq train={len(y_tr)}  val={len(y_va)}  test={len(y_te)}')


---
## 7. AttentionPool + RNN Model Builder


In [ ]:
class AttentionPool(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score = layers.Dense(1)
        self.supports_masking = True

    def call(self, x, mask=None):
        scores = self.score(x)
        if mask is not None:
            mask_f = tf.cast(mask, tf.float32)[..., tf.newaxis]
            scores = scores + (1.0 - mask_f) * -1e9
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(x * weights, axis=1)

    def compute_mask(self, inputs, mask=None):
        return None


def build_attention_rnn(rnn_type, seq_len, n_seq_feat, n_static_feat, n_players,
                        rnn_units=128, player_emb_dim=8,
                        dropout=0.15, rnn_dropout=0.10,
                        huber_delta=1.0, lr=2e-4, wd=1e-4):
    RNNCell = LSTM if rnn_type == 'LSTM' else GRU

    seq_input = Input(shape=(seq_len, n_seq_feat), name='seq_input')
    x = Masking(mask_value=0.0)(seq_input)
    x = Bidirectional(RNNCell(rnn_units, return_sequences=True,
                              dropout=rnn_dropout, recurrent_dropout=rnn_dropout))(x)
    x = AttentionPool()(x)
    x = LayerNormalization()(x)
    x = Dropout(dropout)(x)

    pid_input = Input(shape=(), dtype='int32', name='pid_input')
    p = Embedding(n_players, player_emb_dim,
                  embeddings_regularizer=regularizers.l2(1e-5))(pid_input)
    p = Flatten()(p)

    static_input = Input(shape=(n_static_feat,), name='static_input')
    s = Dense(64, activation='relu')(static_input)
    s = LayerNormalization()(s)
    s = Dropout(dropout)(s)

    merged = Concatenate()([x, p, s])
    merged = Dense(96, activation='relu')(merged)
    merged = LayerNormalization()(merged)
    merged = Dropout(dropout * 0.5)(merged)
    output = Dense(1)(merged)

    model = Model(inputs=[seq_input, pid_input, static_input], outputs=output)
    opt = optimizers.AdamW(learning_rate=lr, weight_decay=wd)
    model.compile(optimizer=opt, loss=losses.Huber(delta=huber_delta), metrics=['mae'])
    return model


_m = build_attention_rnn('GRU', SEQ_LEN, n_seq_features, n_static_features, n_players)
print(f'Attention RNN params: {_m.count_params():,}')
del _m


---
## 8. Sequence Models


In [ ]:
BATCH_SIZE = 32
EPOCHS     = 500
PATIENCE   = 50

def train_attention_rnn(rnn_type):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

    model = build_attention_rnn(
        rnn_type, SEQ_LEN, n_seq_features, n_static_features, n_players,
    )

    cbs = [
        callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=15, min_lr=1e-6),
    ]

    t0 = time.time()
    model.fit(
        {'seq_input': X_seq_tr, 'pid_input': X_pid_tr, 'static_input': X_static_tr},
        y_tr_log,
        sample_weight=sw_tr_seq,
        validation_data=(
            {'seq_input': X_seq_va, 'pid_input': X_pid_va, 'static_input': X_static_va},
            y_va_log,
        ),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=cbs, verbose=0,
    )
    elapsed = time.time() - t0

    pred_va = log1p_to_orig(model.predict(
        {'seq_input': X_seq_va, 'pid_input': X_pid_va, 'static_input': X_static_va},
        verbose=0,
    ).flatten())
    pred_te = log1p_to_orig(model.predict(
        {'seq_input': X_seq_te, 'pid_input': X_pid_te, 'static_input': X_static_te},
        verbose=0,
    ).flatten())
    return model, pred_va, pred_te, elapsed


### 8.1 BiGRU + AttentionPool + PlayerEmbed (log1p)

Najbolji sekvencni model u projektu (`WR_RNN_Attention_PlayerEmbed_v2`, R2=0.3262).


In [ ]:
gru_model, gru_val_pred, gru_test_pred, gru_secs = train_attention_rnn('GRU')
log_result('BiGRU + Attention + Embed',
           eval_preds(y_va, gru_val_pred),
           eval_preds(y_te, gru_test_pred),
           gru_secs)


### 8.2 BiLSTM + AttentionPool + PlayerEmbed (log1p)


In [ ]:
lstm_model, lstm_val_pred, lstm_test_pred, lstm_secs = train_attention_rnn('LSTM')
log_result('BiLSTM + Attention + Embed',
           eval_preds(y_va, lstm_val_pred),
           eval_preds(y_te, lstm_test_pred),
           lstm_secs)


---
## 9. Final Comparison Table


In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Model':     name,
        'Val MAE':   round(r['val']['MAE'],   3),
        'Test MAE':  round(r['test']['MAE'],  3),
        'Test RMSE': round(r['test']['RMSE'], 3),
        'Test R2':   round(r['test']['R2'],   4),
        'Seconds':   round(r['seconds'], 1) if r.get('seconds') else None,
    })
summary = pd.DataFrame(rows).sort_values('Test MAE').reset_index(drop=True)
print(summary.to_string(index=False))


In [ ]:
os.makedirs('../results', exist_ok=True)
summary.to_csv('../results/wr_final_models_summary.csv', index=False)
print('Saved: ../results/wr_final_models_summary.csv')
